In [ ]:
# Install everything we need
# streamlit = builds the web app
# pyngrok   = gives us a free public link from Colab

!pip install streamlit pyngrok -q

print("Installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.4 MB/s eta 0:00:00
Installed!


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

print("Upload model.pkl and features.pkl now...")
uploaded = files.upload()

print("\nFiles received:")
for f in uploaded.keys():
    print(f"  ✅ {f}")

Upload model.pkl and features.pkl now...


Saving features.pkl to features.pkl
Saving model.pkl to model.pkl

Files received:
  ✅ features.pkl
  ✅ model.pkl


In [ ]:

# Save uploaded files to Colab's local storage
# so the Streamlit app can read them

import os

for filename, content in uploaded.items():
    with open(filename, 'wb') as f:
        f.write(content)

print("Files saved to Colab disk:")
print("  model.pkl    →", os.path.exists('model.pkl'))
print("  features.pkl →", os.path.exists('features.pkl'))

Files saved to Colab disk:
  model.pkl    → True
  features.pkl → True


In [ ]:

# Write the entire Streamlit app as a Python file
# %%writefile app.py saves everything below into a file called app.py

%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import pickle

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="TrendLens",
    page_icon="🔍",
    layout="centered"
)

# ── Load model and features ───────────────────────────────────
@st.cache_resource
def load_model():
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('features.pkl', 'rb') as f:
        features = pickle.load(f)
    return model, features

model, features = load_model()

# ── Helper maps (must match Notebook 03 encoding) ────────────
DEPT_MAP = {
    'Bottoms': 0, 'Dresses': 1, 'Intimate': 2,
    'Jackets': 3, 'Tops': 4, 'Trend': 5
}
DIVISION_MAP = {
    'General': 0, 'General Petite': 1, 'Initmates': 2
}
CLASS_MAP = {
    'Blouses': 0, 'Bottoms': 1, 'Casual bottoms': 2,
    'Chemises': 3, 'Dresses': 4, 'Fine gauge': 5,
    'Intimates': 6, 'Jackets': 7, 'Jeans': 8,
    'Knits': 9, 'Layering': 10, 'Legwear': 11,
    'Lounge': 12, 'Outerwear': 13, 'Pants': 14,
    'Shorts': 15, 'Skirts': 16, 'Sleep': 17,
    'Sweaters': 18, 'Swimwear': 19, 'Trend': 20
}
AGE_GROUP_MAP = {
    '18-25 (Gen-Z)': 1, '26-35 (Millennials)': 2,
    '36-45': 3, '46-60': 4, '60+': 5
}
SENTIMENT_MAP = {'positive': 1, 'neutral': 0, 'negative': -1}

# ── Header ────────────────────────────────────────────────────
st.title("🔍 TrendLens")
st.subheader("Will this product go viral?")
st.markdown("Fill in the product details below and get an instant prediction.")
st.divider()

# ── Input form ────────────────────────────────────────────────
col1, col2 = st.columns(2)

with col1:
    rating = st.slider("⭐ Product Rating", 1, 5, 4)
    age    = st.number_input("👤 Reviewer Age", min_value=18,
                              max_value=100, value=30)
    dept   = st.selectbox("🏬 Department", list(DEPT_MAP.keys()))
    division = st.selectbox("📦 Division", list(DIVISION_MAP.keys()))

with col2:
    cls         = st.selectbox("👗 Product Class", list(CLASS_MAP.keys()))
    age_group   = st.selectbox("📊 Age Group", list(AGE_GROUP_MAP.keys()))
    sentiment   = st.selectbox("💬 Review Sentiment",
                               ['positive', 'neutral', 'negative'])
    feedback_count = st.number_input("👍 Positive Feedback Count",
                                     min_value=0, max_value=500, value=10)

review_len = st.slider("📝 Review Length (words)", 5, 200, 50)

st.divider()

# ── Predict button ────────────────────────────────────────────
if st.button("🔮 Predict Viral Score", use_container_width=True):

    # Build the feature row exactly as the model expects
    max_feedback = 500.0   # same max used in Notebook 03

    input_data = {
        'Rating'            : rating,
        'Age'               : age,
        'sentiment_score'   : 0.5 if sentiment == 'positive'
                              else -0.3 if sentiment == 'negative'
                              else 0.0,
        'sentiment_encoded' : SENTIMENT_MAP[sentiment],
        'popularity_score'  : min(feedback_count / max_feedback, 1.0),
        'review_length'     : review_len,
        'department_encoded': DEPT_MAP[dept],
        'class_encoded'     : CLASS_MAP[cls],
        'division_encoded'  : DIVISION_MAP[division],
        'age_group_encoded' : AGE_GROUP_MAP[age_group],
        'high_feedback'     : 1 if feedback_count > 25 else 0
    }

    input_df = pd.DataFrame([input_data])[features]

    prediction   = model.predict(input_df)[0]
    probability  = model.predict_proba(input_df)[0][1]
    viral_pct    = round(probability * 100, 1)

    # ── Result display ────────────────────────────────────────
    st.markdown("### 📊 Prediction Result")

    if prediction == 1:
        st.success(f"✅ **VIRAL!** — This product is likely to be recommended.")
    else:
        st.error(f"❌ **NOT VIRAL** — This product may not get recommended.")

    st.metric(label="Viral Probability", value=f"{viral_pct}%")
    st.progress(int(viral_pct))

    st.markdown("---")

    # Show what was entered as a summary
    st.markdown("**📋 Inputs used for this prediction:**")
    summary = {
        "Rating": rating, "Age": age, "Department": dept,
        "Sentiment": sentiment, "Feedback Count": feedback_count,
        "Review Length": review_len
    }
    st.table(pd.DataFrame(summary, index=["Value"]).T)

Writing app.py


In [ ]:
from pyngrok import ngrok

# Paste your ngrok token here 👇
NGROK_AUTH_TOKEN = "3C7AwJqn6Rde7ENuimTHaa4Zk1C_9e7fn1aAELxjCx9MySkT" # <--- IMPORTANT: YOU MUST REPLACE THIS PLACEHOLDER WITH YOUR ACTUAL NGROK TOKEN!

if NGROK_AUTH_TOKEN == "YOUR_TOKEN_HERE":
    print("\n===========================================================")
    print("  🚨 WARNING: ngrok authentication token is NOT set! 🚨")
    print("===========================================================")
    print("  Please replace 'YOUR_TOKEN_HERE' with your actual ngrok authentication token.")
    print("  You can find your token at: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("===========================================================\n")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok token set!")

ngrok token set!


In [ ]:

# Launch the Streamlit app and get your public link!

import subprocess, threading, time

# Start Streamlit in the background
def run_app():
    subprocess.run(["streamlit", "run", "app.py",
                    "--server.port", "8501",
                    "--server.headless", "true"])

thread = threading.Thread(target=run_app, daemon=True)
thread.start()

# Wait a moment for app to start
time.sleep(5)

# Open public tunnel to port 8501
public_url = ngrok.connect(8501)

print("=" * 50)
print("  🚀 TrendLens is LIVE!")
print("=" * 50)
print(f"  Open this link → {public_url}")
print("=" * 50)
print("  Share this link with anyone — it works!")
print("  (Link stays active while this cell runs)")

  🚀 TrendLens is LIVE!
  Open this link → NgrokTunnel: "https://fiducially-nettlelike-marquita.ngrok-free.dev" -> "http://localhost:8501"
  Share this link with anyone — it works!
  (Link stays active while this cell runs)
